In [11]:
import os
import json
import numpy as np
import re
from collections import defaultdict, deque
from scipy.stats import mannwhitneyu

# =========================
# PATHS
# =========================
pred_dir = "/home/comp/24483737/fc/eval/all-steer-merged"
score_dir = "/home/comp/24483737/fc/eval/res/last/all-steer"

metrics = ["Misleadingness", "Informativeness", "Soundness", "Readability"]

# =========================
# helpers
# =========================
def clean_score(score_str):

    if not isinstance(score_str, str):
        return None

    match = re.search(r'\{[\s\S]*?\}', score_str)
    if not match:
        return None

    s = match.group()

    s = re.sub(
        r'([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:',
        r'\1"\2":',
        s
    )

    s = s.replace("'", '"')

    try:
        return json.loads(s)
    except:
        return None


def cliffs_delta(a, b):
    greater = sum(x > y for x in a for y in b)
    less = sum(x < y for x in a for y in b)
    return (greater - less) / (len(a) * len(b))


def bootstrap_ci(data, n=2000):
    means = []
    for _ in range(n):
        sample = np.random.choice(data, len(data), replace=True)
        means.append(np.mean(sample))
    return np.percentile(means, [2.5, 97.5])


def extract_verdict(pred):

    m = re.search(
        r'verdict:\s*([a-z\- ]+?)(?:\.|\n|$)',
        pred.lower()
    )

    if not m:
        return None

    return m.group(1).strip()


def verdict_correct(label, pred):

    label = label.lower().strip()
    verdict = extract_verdict(pred)

    if verdict is None:
        return False

    return verdict == label


# =========================
# STEP 1 — load predictions
# =========================
# store ALL correctness results, not overwrite
correct_map = defaultdict(deque)

total_preds = 0

for fname in os.listdir(pred_dir):

    if not fname.endswith(".json"):
        continue

    typ = "kv" if "hallu" in fname else "iv"

    with open(os.path.join(pred_dir, fname)) as f:

        for line in f:
            try:
                row = json.loads(line)
            except:
                continue

            qid = row.get("question_id")

            pred_text = ""
            if "predicted" in row:
                pred_text = row["predicted"]
            elif "choices" in row:
                pred_text = row["choices"][0]["turns"][0]

            label = row.get("label", "")

            if qid:

                is_correct = verdict_correct(label, pred_text)

                correct_map[(typ, qid)].append(is_correct)
                total_preds += 1

print("Loaded prediction rows:", total_preds)
print("Unique (typ,qid):", len(correct_map))


# =========================
# STEP 2 — score buckets
# =========================
def make_bucket():
    return {m: [] for m in metrics}

kv_correct = make_bucket()
kv_wrong = make_bucket()
iv_correct = make_bucket()
iv_wrong = make_bucket()

skipped = 0
matched = 0

for fname in os.listdir(score_dir):

    if not fname.endswith(".json"):
        continue

    typ = "kv" if "hallu" in fname else "iv"

    with open(os.path.join(score_dir, fname)) as f:

        for line in f:

            try:
                row = json.loads(line)
            except:
                skipped += 1
                continue

            score_obj = clean_score(row.get("score"))
            if score_obj is None:
                skipped += 1
                continue

            qid = row.get("id")

            key = (typ, qid)

            if key not in correct_map or not correct_map[key]:
                skipped += 1
                continue

            # POP correctness — guarantees 1-to-1 mapping
            is_correct = correct_map[key].popleft()
            matched += 1

            if typ == "kv":
                bucket = kv_correct if is_correct else kv_wrong
            else:
                bucket = iv_correct if is_correct else iv_wrong

            for m in metrics:
                val = score_obj.get(m)
                if isinstance(val, (int, float)):
                    bucket[m].append(val)

print("Matched score rows:", matched)
print("Skipped score rows:", skipped)


# =========================
# STEP 3 — analysis
# =========================
def analyze(groupA, groupB, title):

    print("\n======", title, "======")

    for m in metrics:

        a = np.array(groupA[m])
        b = np.array(groupB[m])

        if len(a) == 0 or len(b) == 0:
            print(m, "⚠ no data")
            continue

        u, p = mannwhitneyu(a, b)
        delta = cliffs_delta(a, b)

        print(f"\n{m}")
        print("KV mean:", a.mean(), bootstrap_ci(a))
        print("IV mean:", b.mean(), bootstrap_ci(b))
        print("p =", p)
        print("delta =", delta)


analyze(kv_correct, iv_correct, "Correct Predictions")
analyze(kv_wrong, iv_wrong, "Wrong Predictions")



Skipped broken rows: 8
Loaded samples:
Misleadingness KV: 2621 IV: 799
Informativeness KV: 2621 IV: 799
Soundness KV: 2621 IV: 799
Readability KV: 2621 IV: 799

===== KV vs IV Analysis =====



Misleadingness
KV mean = 1.58 CI[1.54673789 1.62190004]
IV mean = 1.90 CI[1.81226533 1.99123905]
Mann–Whitney p = 0.000000
Cliff's delta = -0.130
------
Informativeness
KV mean = 4.56 CI[4.5333842  4.57802366]
IV mean = 4.86 CI[4.82850438 4.88360451]
Mann–Whitney p = 0.000000
Cliff's delta = -0.277
------
Soundness
KV mean = 4.72 CI[4.69667112 4.74399084]
IV mean = 4.79 CI[4.74718398 4.82853567]
Mann–Whitney p = 0.000108
Cliff's delta = -0.062
------
Readability
KV mean = 4.85 CI[4.83250668 4.86264784]
IV mean = 4.79 CI[4.75841677 4.82478098]
Mann–Whitney p = 0.004991
Cliff's delta = 0.040
------


In [16]:
import os
import json
import numpy as np
import re
from collections import deque
from scipy.stats import mannwhitneyu

# =========================
# PATHS
# =========================
pred_dir = "/home/comp/24483737/fc/eval/all-steer-merged"
score_dir = "/home/comp/24483737/fc/eval/res/last/all-steer"

metrics = ["Misleadingness", "Informativeness", "Soundness", "Readability"]

# =========================
# helpers
# =========================
def clean_score(score_str):

    if not isinstance(score_str, str):
        return None

    match = re.search(r'\{[\s\S]*?\}', score_str)
    if not match:
        return None

    s = match.group()

    s = re.sub(
        r'([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:',
        r'\1"\2":',
        s
    )

    s = s.replace("'", '"')

    try:
        return json.loads(s)
    except:
        return None


def cliffs_delta(a, b):
    greater = sum(x > y for x in a for y in b)
    less = sum(x < y for x in a for y in b)
    return (greater - less) / (len(a) * len(b))


def bootstrap_ci(data, n=2000):
    means = []
    for _ in range(n):
        sample = np.random.choice(data, len(data), replace=True)
        means.append(np.mean(sample))
    return np.percentile(means, [2.5, 97.5])


def extract_verdict(pred):

    m = re.search(
        r'verdict:\s*([a-z\- ]+?)(?:\.|\n|$)',
        pred.lower()
    )

    if not m:
        return None

    return m.group(1).strip()


def verdict_correct(label, pred):

    label = label.lower().strip()
    verdict = extract_verdict(pred)

    if verdict is None:
        return False

    return verdict == label


# =========================
# STEP 1 — load predictions
# =========================
prediction_correctness = []

for fname in sorted(os.listdir(pred_dir)):

    if not fname.endswith(".json"):
        continue

    typ = "kv" if "hallu" in fname else "iv"

    with open(os.path.join(pred_dir, fname)) as f:

        for line in f:
            try:
                row = json.loads(line)
            except:
                continue

            pred_text = row.get("predicted", "")
            label = row.get("label", "")

            is_correct = verdict_correct(label, pred_text)

            prediction_correctness.append((typ, is_correct))

print("Loaded prediction rows:", len(prediction_correctness))

correct_queue = deque(prediction_correctness)


# =========================
# STEP 2 — score buckets
# =========================
def make_bucket():
    return {m: [] for m in metrics}

kv_correct = make_bucket()
kv_wrong = make_bucket()
iv_correct = make_bucket()
iv_wrong = make_bucket()

skipped = 0
matched = 0

for fname in sorted(os.listdir(score_dir)):

    if not fname.endswith(".json"):
        continue

    typ = "kv" if "hallu" in fname else "iv"

    with open(os.path.join(score_dir, fname)) as f:

        for line in f:

            try:
                row = json.loads(line)
            except:
                skipped += 1
                continue

            score_obj = clean_score(row.get("score"))
            if score_obj is None:
                skipped += 1
                continue

            if not correct_queue:
                skipped += 1
                continue

            pred_typ, is_correct = correct_queue.popleft()

            if pred_typ != typ:
                skipped += 1
                continue

            matched += 1

            if typ == "kv":
                bucket = kv_correct if is_correct else kv_wrong
            else:
                bucket = iv_correct if is_correct else iv_wrong

            for m in metrics:
                val = score_obj.get(m)
                if isinstance(val, (int, float)):
                    bucket[m].append(val)

print("Matched rows:", matched)
print("Skipped rows:", skipped)


# =========================
# STEP 3 — analysis
# =========================
def analyze(groupA, groupB, title):

    print("\n======", title, "======")

    for m in metrics:

        a = np.array(groupA[m])
        b = np.array(groupB[m])

        if len(a) == 0 or len(b) == 0:
            print(m, "⚠ no data")
            continue

        u, p = mannwhitneyu(a, b)
        delta = cliffs_delta(a, b)

        print(f"\n{m}")
        print("KV mean:", a.mean(), bootstrap_ci(a))
        print("IV mean:", b.mean(), bootstrap_ci(b))
        print("p =", p)
        print("delta =", delta)


analyze(kv_correct, iv_correct, "Correct Predictions")
analyze(kv_wrong, iv_wrong, "Wrong Predictions")


Loaded prediction rows: 3422
Matched rows: 3422
Skipped rows: 0

====== Correct Predictions ======

Misleadingness
KV mean: 1.5151682294539437 [1.47214562 1.55874242]
IV mean: 1.8920570264765784 [1.78207739 2.01221996]
p = 7.6745101608728e-12
delta = -0.16920565771307697

Informativeness
KV mean: 4.56315499172642 [4.53666575 4.59185052]
IV mean: 4.853360488798371 [4.81873727 4.89002037]
p = 9.215530225376924e-28
delta = -0.26521962338081045

Soundness
KV mean: 4.740761169332598 [4.71318257 4.76889134]
IV mean: 4.788187372708758 [4.7311609  4.83910387]
p = 0.023490643566523438
delta = -0.04493008740899343

Readability
KV mean: 4.8499724214009925 [4.83177055 4.86872587]
IV mean: 4.796334012219959 [4.75560081 4.83706721]
p = 0.022408094269390947
delta = 0.041003928405732304

====== Wrong Predictions ======

Misleadingness
KV mean: 1.738861386138614 [1.66581064 1.81559406]
IV mean: 1.9123376623376624 [1.76623377 2.06168831]
p = 0.24953753331114836
delta = -0.04025893660794651

Informativen